# YouTube Matching Pipeline (yt-dlp)

**Goal:** Find the OFFICIAL YouTube video for each Spotify track

**Method:** Use yt-dlp's built-in search (no API quotas!)

**Output:** `youtube_matches.csv` with video IDs and confidence scores

---

## Pipeline Flow

1. Load Spotify dataset
2. Search YouTube for both 'music video' and 'official audio'
3. Score matches based on:
   - **Title match** (track name + artist in title)
   - **Channel match** (artist name in channel - MOST IMPORTANT)
4. Prioritize official artist channels over random uploads
5. Save results with confidence scores
6. Resume capability for interrupted runs

**Key Strategy:** Official artist channel > everything else
- "Hold On" audio by Chord Overstreet (official) > "Hold On" video by RandomChannel

## 1. Setup & Imports

In [2]:
import pandas as pd
import numpy as np
import re
import json
from pathlib import Path
from tqdm.auto import tqdm
import subprocess
import time

## 2. Configuration

In [3]:
# Paths
SPOTIFY_PATH = "../data/raw/spotify_tracks.csv"
OUTPUT_PATH = "../data/raw/youtube_matches.csv"

# Processing settings
BATCH_SIZE = 100  # Process in batches for resumability
MAX_SEARCH_RESULTS = 5  # How many YouTube results to consider per song
SLEEP_BETWEEN_SEARCHES = 0.5  # Seconds to wait between searches (be nice to YouTube)

# Confidence thresholds
HIGH_CONFIDENCE = 0.8  # Strong match
MEDIUM_CONFIDENCE = 0.6  # Decent match
LOW_CONFIDENCE = 0.4  # Weak match

## 3. Load Spotify Dataset

In [4]:
# Load original Spotify dataset
spotify_df = pd.read_csv(SPOTIFY_PATH)

print(f"Total Spotify tracks: {len(spotify_df):,}")
print(f"\nColumns: {list(spotify_df.columns)}")

# Show sample
spotify_df[["track_id", "track_name", "artists", "popularity"]].head(10)

Total Spotify tracks: 114,000

Columns: ['Unnamed: 0', 'track_id', 'artists', 'album_name', 'track_name', 'popularity', 'duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature', 'track_genre']


,track_id,track_name,artists,popularity
0,5SuOikwiRyPMVoIQDJUgSV,Comedy,Gen Hoshino,73
1,4qPNDBW1i3p13qLCt0Ki3A,Ghost - Acoustic,Ben Woodward,55
2,1iJBSr7s7jYXzM8EGcbK5b,To Begin Again,Ingrid Michaelson;ZAYN,57
3,6lfxq3CG4xtTiEg7opyCyx,Can't Help Falling In Love,Kina Grannis,71
4,5vjLSffimiIP26QG5WcN2K,Hold On,Chord Overstreet,82
5,01MVOl9KtVTNfFiBU9I7dc,Days I Will Remember,Tyrone Wells,58
6,6Vc5wAMmXdKIAM7WUoEb7N,Say Something,A Great Big World;Christina Aguilera,74
7,1EzrEOXmMH3G43AXT1y7pA,I'm Yours,Jason Mraz,80
8,0IktbUcnAGrvD03AWnz3Q8,Lucky,Jason Mraz;Colbie Caillat,74
9,7k9GuJYLp2AzqokyEdwEw2,Hunger,Ross Copperman,56


## 4. Core Functions

In [5]:
def normalize_text(text):
    """
    Normalize text for matching (remove special chars, lowercase, clean up)
    """
    if pd.isna(text):
        return ""
    
    text = str(text).lower()
    
    # Remove common noise patterns
    text = re.sub(r"\(.*?\)", " ", text)  # Remove parentheses content
    text = re.sub(r"\[.*?\]", " ", text)  # Remove brackets content
    
    # Remove common YouTube keywords
    noise_words = [
        "official", "video", "audio", "lyrics", "lyric",
        "music", "hd", "hq", "mv", "visualizer", "visualiser",
        "explicit", "clean", "version"
    ]
    for word in noise_words:
        text = re.sub(rf"\b{word}\b", " ", text)
    
    # Remove special characters (keep alphanumeric and spaces)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    
    # Collapse multiple spaces
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

In [6]:
def calculate_match_score(track_name, artist_name, youtube_title, youtube_channel):
    """
    Calculate confidence score (0-1) for how well YouTube video matches the song
    
    Scoring factors:
    1. Title match (track name + artist in title)
    2. Channel match (is it the artist's official channel?)
    3. Content type bonus (official audio/video > lyric video > variations)
    4. Variation penalty (acoustic, remix, live, cover versions get penalized)
    5. Non-official channel soft penalty (random channels slightly penalized)
    """
    # Normalize all text
    track_norm = normalize_text(track_name)
    artist_norm = normalize_text(artist_name)
    title_norm = normalize_text(youtube_title)
    channel_norm = normalize_text(youtube_channel)
    
    # Check content in the ORIGINAL title (before normalization)
    title_lower = youtube_title.lower()
    channel_lower = youtube_channel.lower()
    
    # Preferred content types (BEST - no penalty, gets bonus)
    preferred_content = [
        'official audio',
        'official video', 
        'official music video',
        'audio oficial',  # Spanish
        'música oficial'   # Spanish
    ]
    has_preferred = any(keyword in title_lower for keyword in preferred_content)
    
    # Secondary content types (OKAY - small penalty)
    secondary_content = [
        'lyric video',
        'lyrics',
        'lyric',
        'visualizer',
        'visualiser',
        'letra',  # Spanish for lyrics
    ]
    has_secondary = any(keyword in title_lower for keyword in secondary_content)
    
    # Variation keywords (BAD - heavy penalty)
    variation_keywords = [
        'acoustic', 'remix', 'remix)', 'live', 'cover', 'karaoke',
        'instrumental', 'acapella', 'stripped', 'demo',
        'radio edit', 'extended', 'slowed', 'sped up',
        'reverb', 'bass boosted', 'nightcore', 'mashup',
        'remaster', 'remastered', 'piano', 'guitar'
    ]
    has_variation = any(keyword in title_lower for keyword in variation_keywords)
    
    # Label/Official channel indicators (these are GOOD channels even without artist name)
    label_indicators = [
        'vevo', 'records', 'recording', 'music', 'entertainment',
        'official', 'label', 'studio', 'media', 'productions',
        'hybe', 'yg', 'sm', 'jyp',  # K-pop labels
        'def jam', 'columbia', 'atlantic', 'warner', 'universal', 'sony',  # Major labels
        'topic'  # YouTube auto-generated official channels
    ]
    has_label_indicator = any(indicator in channel_lower for indicator in label_indicators)
    
    # Split into tokens
    track_tokens = set(track_norm.split())
    artist_tokens = set(artist_norm.split())
    title_tokens = set(title_norm.split())
    channel_tokens = set(channel_norm.split())
    
    # Calculate title matches
    track_matches = len(track_tokens & title_tokens)
    artist_in_title = len(artist_tokens & title_tokens)
    
    # Calculate channel match (is artist name in channel?)
    artist_in_channel = len(artist_tokens & channel_tokens)
    
    # Determine if channel is likely official
    is_likely_official = (artist_in_channel > 0) or has_label_indicator
    
    # Base scores
    track_score = track_matches / max(len(track_tokens), 1)
    artist_title_score = artist_in_title / max(len(artist_tokens), 1)
    artist_channel_score = artist_in_channel / max(len(artist_tokens), 1)
    
    # Weighted combination
    # - Track name match: 40%
    # - Artist in title: 20%
    # - Artist in channel: 40% (MOST IMPORTANT - official channel bonus)
    confidence = (track_score * 0.4) + (artist_title_score * 0.2) + (artist_channel_score * 0.4)
    
    # Apply content type modifiers
    if has_preferred:
        # Boost official audio/video by 20%
        confidence = min(confidence * 1.2, 1.0)
    elif has_secondary:
        # Slight penalty for lyric videos (10%)
        confidence = confidence * 0.9
    
    # Apply variation penalty (heavy - reduce score by 50%)
    if has_variation:
        confidence = confidence * 0.5
    
    # Apply soft non-official channel penalty
    # Only penalize if: (1) not official/label channel AND (2) low artist match in channel
    if not is_likely_official:
        # Gentle 15% penalty for random channels
        confidence = confidence * 0.85
    
    return confidence

In [7]:
def search_youtube_ytdlp(track_name, artist_name, max_results=5, prefer_video=True):
    """
    Search YouTube using yt-dlp and return top results with metadata
    
    Strategy:
    1. First try to find official music video
    2. If no good match, fall back to official audio
    """
    # Build search query - prioritize music video
    if prefer_video:
        query = f"{track_name} {artist_name} official music video"
    else:
        query = f"{track_name} {artist_name} official audio"
    
    try:
        # Run yt-dlp search
        cmd = [
            "yt-dlp",
            f"ytsearch{max_results}:{query}",
            "--dump-json",
            "--skip-download",
            "--no-warnings",
            "--quiet"
        ]
        
        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=30
        )
        
        if result.returncode != 0:
            return []
        
        # Parse results (each line is a JSON object)
        results = []
        for line in result.stdout.strip().split('\n'):
            if line.strip():
                try:
                    video_data = json.loads(line)
                    results.append({
                        "video_id": video_data.get("id", ""),
                        "title": video_data.get("title", ""),
                        "channel": video_data.get("uploader", ""),
                        "duration": video_data.get("duration", 0),
                        "view_count": video_data.get("view_count", 0)
                    })
                except json.JSONDecodeError:
                    continue
        
        return results
        
    except subprocess.TimeoutExpired:
        return []
    except Exception as e:
        return []

In [8]:
def find_best_match(track_name, artist_name, max_results=5):
    """
    Search YouTube and return the best match with confidence score
    
    Strategy:
    1. Search for both 'official music video' and 'official audio'
    2. Score all results using title + channel matching
    3. Prioritize official artist channels (high channel match score)
    4. Return best overall match
    """
    all_results = []
    
    # Search for music videos
    video_results = search_youtube_ytdlp(track_name, artist_name, max_results, prefer_video=True)
    if video_results:
        for result in video_results:
            result['search_type'] = 'music_video'
        all_results.extend(video_results)
    
    # Search for official audio
    audio_results = search_youtube_ytdlp(track_name, artist_name, max_results, prefer_video=False)
    if audio_results:
        for result in audio_results:
            result['search_type'] = 'official_audio'
        all_results.extend(audio_results)
    
    if not all_results:
        return None
    
    # Score all results with channel-aware scoring
    scored_results = []
    for result in all_results:
        score = calculate_match_score(
            track_name, 
            artist_name, 
            result["title"],
            result["channel"]
        )
        scored_results.append({
            **result,
            "confidence": score
        })
    
    # Sort by confidence (channel match is heavily weighted)
    scored_results.sort(key=lambda x: x["confidence"], reverse=True)
    
    # Return best match
    best = scored_results[0]
    
    return {
        "youtube_video_id": best["video_id"],
        "youtube_url": f"https://www.youtube.com/watch?v={best['video_id']}",
        "youtube_title": best["title"],
        "youtube_channel": best["channel"],
        "youtube_duration": best["duration"],
        "youtube_views": best["view_count"],
        "match_confidence": round(best["confidence"], 3),
        "content_type": best["search_type"]
    }

## 5. Test on Sample Songs

In [8]:
# Test with a single song
test_track = spotify_df.iloc[4]  # "Hold On" by Chord Overstreet

print(f"Testing: {test_track['track_name']} by {test_track['artists']}")
print("\nSearching YouTube...\n")

match = find_best_match(test_track["track_name"], test_track["artists"])

if match:
    print(f"✓ Found match!")
    print(f"  Video ID: {match['youtube_video_id']}")
    print(f"  Title: {match['youtube_title']}")
    print(f"  Channel: {match['youtube_channel']}")
    print(f"  Content Type: {match['content_type']}")
    print(f"  Confidence: {match['match_confidence']:.1%}")
    print(f"  URL: {match['youtube_url']}")
else:
    print("✗ No match found")

Testing: Hold On by Chord Overstreet

Searching YouTube...

✓ Found match!
  Video ID: a0sxc2jIYD0
  Title: Chord Overstreet - Hold On (Audio)
  Channel: Chord Overstreet
  Content Type: official_audio
  Confidence: 100.0%
  URL: https://www.youtube.com/watch?v=a0sxc2jIYD0


In [9]:
# Test with first 10 songs
test_df = spotify_df.head(10).copy()

test_matches = []

print("Testing first 10 songs...\n")

for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):
    match = find_best_match(row["track_name"], row["artists"])
    
    result = {
        "track_id": row["track_id"],
        "track_name": row["track_name"],
        "artists": row["artists"],
        "video_found": match is not None
    }
    
    if match:
        result.update(match)
    else:
        result.update({
            "youtube_video_id": "",
            "youtube_url": "",
            "youtube_title": "",
            "youtube_channel": "",
            "youtube_duration": 0,
            "youtube_views": 0,
            "match_confidence": 0.0,
            "content_type": ""
        })
    
    test_matches.append(result)
    
    time.sleep(SLEEP_BETWEEN_SEARCHES)

test_results = pd.DataFrame(test_matches)

# Show results
print("\n=== Test Results ===")
print(f"Total: {len(test_results)}")
print(f"Found: {test_results['video_found'].sum()}")
print(f"Not found: {(~test_results['video_found']).sum()}")
print(f"\nConfidence distribution:")
print(f"  High (>{HIGH_CONFIDENCE}): {(test_results['match_confidence'] > HIGH_CONFIDENCE).sum()}")
print(f"  Medium ({MEDIUM_CONFIDENCE}-{HIGH_CONFIDENCE}): {((test_results['match_confidence'] >= MEDIUM_CONFIDENCE) & (test_results['match_confidence'] <= HIGH_CONFIDENCE)).sum()}")
print(f"  Low (<{MEDIUM_CONFIDENCE}): {(test_results['match_confidence'] < MEDIUM_CONFIDENCE).sum()}")
print(f"\nContent type distribution:")
print(f"  Music Videos: {(test_results['content_type'] == 'music_video').sum()}")
print(f"  Official Audio: {(test_results['content_type'] == 'official_audio').sum()}")

test_results[["track_name", "artists", "youtube_title", "content_type", "match_confidence", "video_found"]]

Testing first 10 songs...



  0%|          | 0/10 [00:00<?, ?it/s]


=== Test Results ===
Total: 10
Found: 10
Not found: 0

Confidence distribution:
  High (>0.8): 3
  Medium (0.6-0.8): 6
  Low (<0.6): 1

Content type distribution:
  Music Videos: 8
  Official Audio: 2


,track_name,artists,youtube_title,content_type,match_confidence,video_found
0,Comedy,Gen Hoshino,星野源 – 喜劇 (Official Video),music_video,0.480,True
1,Ghost - Acoustic,Ben Woodward,Ben Woodward - Original,official_audio,0.600,True
2,To Begin Again,Ingrid Michaelson;ZAYN,"Ingrid Michaelson, ZAYN - To Begin Again (Offi...",music_video,0.720,True
3,Can't Help Falling In Love,Kina Grannis,Kina Grannis - Can't Help Falling In Love (Fro...,music_video,1.000,True
4,Hold On,Chord Overstreet,Chord Overstreet - Hold On (Audio),official_audio,1.000,True
5,Days I Will Remember,Tyrone Wells,Days I Will Remember,music_video,0.800,True
6,Say Something,A Great Big World;Christina Aguilera,"A Great Big World, Christina Aguilera - Say So...",music_video,1.000,True
7,I'm Yours,Jason Mraz,Jason Mraz - I'm Yours (Official Video),music_video,0.720,True
8,Lucky,Jason Mraz;Colbie Caillat,Jason Mraz feat. Colbie Caillat - Lucky (Offi...,music_video,0.612,True
9,Hunger,Ross Copperman,Hunger,music_video,0.800,True


## 6. Full Dataset Processing (with Resume Capability)

In [12]:
# Check if we have existing progress
output_path = Path(OUTPUT_PATH)

if output_path.exists():
    print("Found existing youtube_matches.csv")
    existing_matches = pd.read_csv(OUTPUT_PATH)
    print(f"Already processed: {len(existing_matches):,} songs")
    
    # Find songs that still need processing
    processed_ids = set(existing_matches["track_id"].values)
    pending_df = spotify_df[~spotify_df["track_id"].isin(processed_ids)].copy()
    print(f"Remaining: {len(pending_df):,} songs")
else:
    print("No existing progress found. Starting fresh.")
    existing_matches = None
    pending_df = spotify_df.copy()
    print(f"Total to process: {len(pending_df):,} songs")

Found existing youtube_matches.csv
Already processed: 3,700 songs
Remaining: 107,966 songs


In [10]:
# Process in batches with progress saving
def process_batch(df, batch_size=100, start_idx=0):
    """
    Process songs in batches and save progress after each batch
    """
    all_matches = []
    total_songs = len(df)
    
    # Load existing matches if available
    if Path(OUTPUT_PATH).exists():
        existing = pd.read_csv(OUTPUT_PATH)
        all_matches = existing.to_dict('records')
        print(f"Loaded {len(all_matches):,} existing matches\n")
    
    # Process in batches
    for batch_start in range(start_idx, total_songs, batch_size):
        batch_end = min(batch_start + batch_size, total_songs)
        batch_df = df.iloc[batch_start:batch_end]
        
        print(f"\n{'='*60}")
        print(f"Processing batch: {batch_start + 1}-{batch_end} of {total_songs:,}")
        print(f"{'='*60}\n")
        
        batch_matches = []
        
        for idx, row in tqdm(batch_df.iterrows(), total=len(batch_df), desc=f"Batch {batch_start//batch_size + 1}"):
            match = find_best_match(row["track_name"], row["artists"])
            
            result = {
                "track_id": row["track_id"],
                "track_name": row["track_name"],
                "artists": row["artists"],
                "album_name": row["album_name"],
                "popularity": row["popularity"],
                "video_found": match is not None
            }
            
            if match:
                result.update(match)
            else:
                result.update({
                    "youtube_video_id": "",
                    "youtube_url": "",
                    "youtube_title": "",
                    "youtube_channel": "",
                    "youtube_duration": 0,
                    "youtube_views": 0,
                    "match_confidence": 0.0,
                    "content_type": ""
                })
            
            batch_matches.append(result)
            
            time.sleep(SLEEP_BETWEEN_SEARCHES)
        
        # Add batch results to all matches
        all_matches.extend(batch_matches)
        
        # Save progress after each batch
        matches_df = pd.DataFrame(all_matches)
        matches_df.to_csv(OUTPUT_PATH, index=False)
        
        print(f"\n✓ Batch complete. Saved progress: {len(all_matches):,} total matches")
        print(f"  Found in this batch: {sum(m['video_found'] for m in batch_matches)}/{len(batch_matches)}")
    
    return pd.DataFrame(all_matches)

# Note: Set a smaller limit for testing (e.g., 1000 songs)
# Then remove the limit to process all songs
TEST_LIMIT = None  # Set to None to process all songs

if TEST_LIMIT:
    pending_df_limited = pending_df.head(TEST_LIMIT)
    print(f"\n⚠️  TEST MODE: Processing only {TEST_LIMIT:,} songs")
    print(f"To process all songs, set TEST_LIMIT = None\n")
else:
    pending_df_limited = pending_df
    print(f"\n▶ FULL MODE: Processing all {len(pending_df):,} songs\n")


▶ FULL MODE: Processing all 108,403 songs



In [11]:
# Run the processing (comment out this cell until you're ready to run at scale)
# Uncomment the line below when ready:

youtube_matches = process_batch(pending_df_limited, batch_size=BATCH_SIZE)

Loaded 3,500 existing matches


Processing batch: 1-100 of 108,403



Batch 1:   0%|          | 0/100 [00:00<?, ?it/s]


✓ Batch complete. Saved progress: 3,600 total matches
  Found in this batch: 100/100

Processing batch: 101-200 of 108,403



Batch 2:   0%|          | 0/100 [00:00<?, ?it/s]


✓ Batch complete. Saved progress: 3,700 total matches
  Found in this batch: 99/100

Processing batch: 201-300 of 108,403



Batch 3:   0%|          | 0/100 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 7. Analyze Results

In [ ]:
# Load completed matches
if Path(OUTPUT_PATH).exists():
    matches_df = pd.read_csv(OUTPUT_PATH)
    
    print("=== YouTube Matching Results ===")
    print(f"\nTotal processed: {len(matches_df):,}")
    print(f"Videos found: {matches_df['video_found'].sum():,} ({matches_df['video_found'].mean():.1%})")
    print(f"Not found: {(~matches_df['video_found']).sum():,}")
    
    print(f"\n=== Confidence Distribution ===")
    found_df = matches_df[matches_df['video_found']]
    print(f"High confidence (>{HIGH_CONFIDENCE}): {(found_df['match_confidence'] > HIGH_CONFIDENCE).sum():,} ({(found_df['match_confidence'] > HIGH_CONFIDENCE).mean():.1%})")
    print(f"Medium confidence ({MEDIUM_CONFIDENCE}-{HIGH_CONFIDENCE}): {((found_df['match_confidence'] >= MEDIUM_CONFIDENCE) & (found_df['match_confidence'] <= HIGH_CONFIDENCE)).sum():,}")
    print(f"Low confidence (<{MEDIUM_CONFIDENCE}): {(found_df['match_confidence'] < MEDIUM_CONFIDENCE).sum():,}")
    
    print(f"\n=== Content Type Distribution ===")
    print(f"Music Videos: {(found_df['content_type'] == 'music_video').sum():,} ({(found_df['content_type'] == 'music_video').mean():.1%})")
    print(f"Official Audio: {(found_df['content_type'] == 'official_audio').sum():,} ({(found_df['content_type'] == 'official_audio').mean():.1%})")
    
    print(f"\n=== Sample High Confidence Matches ===")
    high_conf = found_df[found_df['match_confidence'] > HIGH_CONFIDENCE].head(10)
    display(high_conf[["track_name", "artists", "youtube_title", "content_type", "match_confidence", "youtube_views"]])
    
    print(f"\n=== Sample Low Confidence Matches ===")
    low_conf = found_df[found_df['match_confidence'] < MEDIUM_CONFIDENCE].head(10)
    display(low_conf[["track_name", "artists", "youtube_title", "match_confidence"]])
else:
    print("No results file found yet. Run the processing cell first.")

## 8. Next Steps

✓ **You now have:** `youtube_matches.csv` with video IDs and confidence scores

**Next notebook:** `04b_youtube_audio_download.ipynb`
- Use the video IDs to download `.wav` files
- Save to `data/raw/youtube_audio/`

**Then:** `04c_youtube_audio_features.ipynb`
- Extract librosa features from downloaded audio
- Your existing feature extraction pipeline